# Kalpa Health · Brief A · Part 2: Analysis (H1 to H5)

Reads the clean tables saved by `Part 1: Cleaning`. No cleaning happens here.

Cells marked `# ADDED` are new; everything else is your original code, in your original order.

In [ ]:
import pandas as pd
import numpy as np
import os

# ADDED: same folder as OUT_DIR in the cleaning notebook
DATA_DIR = r"/content/drive/MyDrive/Kalpa healthcare"

h1_data = pd.read_pickle(os.path.join(DATA_DIR, "h1_data.pkl"))
df_plan = pd.read_pickle(os.path.join(DATA_DIR, "df_plan.pkl"))
df_patients = pd.read_pickle(os.path.join(DATA_DIR, "df_patients.pkl"))
print(h1_data.shape, df_plan.shape, df_patients.shape)


Step 4: Check for H1

In [ ]:
#get metric from the data
test_volume=h1_data.groupby('year').agg(total_patients = ('patient_id','nunique'),
                                        total_test = ('is_test','sum'),
                                        total_revenue = ('test_price','sum'),
                                        total_visits = ('booking_id','nunique')).reset_index()
test_volume

In [ ]:
# Total tests in each year
old = test_volume.iloc[0]['total_test']
new = test_volume.iloc[1]['total_test']

# Calculate percentage increase
gap = ((new - old) / old) * 100

print('Increase by percent:', round(gap, 2), '%')

# H2 - Which branch is short?

In [ ]:
test_volume['test_per_visit'] = test_volume['total_test']/test_volume['total_visits']
test_volume['visit_per_patient'] = test_volume['total_visits']/test_volume['total_patients']
test_volume

In [ ]:
# H1 - Calculate percentage change in metrics

# Get H1 2025 and H1 2026 values
test_per_visit_2025 = test_volume.iloc[0]["test_per_visit"]

test_per_visit_2026 = test_volume.iloc[1]["test_per_visit"]

visit_per_patient_2025 = test_volume.iloc[0]["visit_per_patient"]

visit_per_patient_2026 = test_volume.iloc[1]["visit_per_patient"]

patients_2025 = test_volume.iloc[0]["total_patients"]

patients_2026 = test_volume.iloc[1]["total_patients"]


# Calculate percentage change
test_per_visit_change = ((test_per_visit_2026 - test_per_visit_2025) / test_per_visit_2025) * 100

visit_per_patient_change = ((visit_per_patient_2026 - visit_per_patient_2025)/ visit_per_patient_2025) * 100

patients_change = ((patients_2026 - patients_2025)/patients_2025) * 100

# Display results
print(f"Test per visit change: {test_per_visit_change:.1f}%")
print(f"Visit per patient change: {visit_per_patient_change:.1f}%")
print(f"Patients change: {patients_change:.1f}%")

# H3 - Where is it short?
 Drill down by city and channel

For city

In [ ]:
test_volume_city = h1_data.groupby(['year','city']).agg(total_patients = ('patient_id','nunique'),
                                        total_test = ('is_test','sum'),
                                        total_revenue = ('test_price','sum'),
                                        total_visits = ('booking_id','nunique')).reset_index()
test_volume_city

In [ ]:
# Calculate visits per patient and tests per visit

test_volume_city["test_per_visit"] = (test_volume_city["total_test"] / test_volume_city["total_visits"])
test_volume_city["visit_per_patient"] = (test_volume_city["total_visits"] / test_volume_city["total_patients"])

# Display the summary
test_volume_city

In [ ]:
# H3 - Compare cities between 2025 and 2026

# Make sure year is numeric
test_volume_city["year"] = pd.to_numeric(test_volume_city["year"], errors="coerce")

# Filter each year again
city_2025 = test_volume_city[test_volume_city["year"] == 2025]
city_2026 = test_volume_city[test_volume_city["year"] == 2026]

print(city_2025)
print(city_2026)

In [ ]:
# Use city as the index so matching cities line up
city_2025 = city_2025.set_index("city")
city_2026 = city_2026.set_index("city")

city_comparison = pd.DataFrame({
    "H1 2025 tests": city_2025["total_test"],
    "H1 2026 tests": city_2026["total_test"],
    "H1 2025 patients": city_2025["total_patients"],
    "H1 2026 patients": city_2026["total_patients"],
    "H1 2025 V/P": city_2025["visit_per_patient"],
    "H1 2026 V/P": city_2026["visit_per_patient"],
    "H1 2025 T/V": city_2025["test_per_visit"],
    "H1 2026 T/V": city_2026["test_per_visit"]
})

city_comparison

In [ ]:
result_city = pd.DataFrame()
result_city['H1 2025 tests'] = city_comparison['H1 2025 tests']
result_city['H1 2026 tests'] = city_comparison['H1 2026 tests']

# Calculate percentage growth in tests
result_city["Growth"] = ((city_comparison["H1 2026 tests"] - city_comparison["H1 2025 tests"])/ city_comparison["H1 2025 tests"] * 100).round(1)

# Calculate percentage change in patients
result_city["Patients"] = ((city_comparison["H1 2026 patients"] - city_comparison["H1 2025 patients"])/ city_comparison["H1 2025 patients"] * 100).round(1)

# Calculate percentage change in V/P
result_city["V/P"] = ((city_comparison["H1 2026 V/P"] - city_comparison["H1 2025 V/P"])/ city_comparison["H1 2025 V/P"] * 100).round(1)

# Calculate percentage change in T/V
result_city["T/V"] = ((city_comparison["H1 2026 T/V"] - city_comparison["H1 2025 T/V"]) / city_comparison["H1 2025 T/V"] * 100).round(1)

result_city

For Channel

For Channel

In [ ]:
test_volume_channel = h1_data.groupby(['year','channel']).agg(total_patients = ('patient_id','nunique'),
                                        total_test = ('is_test','sum'),
                                        total_revenue = ('test_price','sum'),
                                        total_visits = ('booking_id','nunique')).reset_index()

In [ ]:
# Calculate visits per patient and tests per visit

test_volume_channel["test_per_visit"] = (test_volume_channel["total_test"] / test_volume_channel["total_visits"])
test_volume_channel["visit_per_patient"] = (test_volume_channel["total_visits"] / test_volume_channel["total_patients"])

test_volume_channel

In [ ]:
# Make sure year is numeric
test_volume_channel["year"] = pd.to_numeric(test_volume_channel["year"], errors="coerce")

# Filter each year again
channel_2025 = test_volume_channel[test_volume_channel["year"] == 2025]
channel_2026 = test_volume_channel[test_volume_channel["year"] == 2026]

print(channel_2025)
print(channel_2026)

In [ ]:
# Use channel as the index so matching channel line up
channel_2025 = channel_2025.set_index("channel")
channel_2026 = channel_2026.set_index("channel")

channel_comparison = pd.DataFrame({
    "H1 2025 tests": channel_2025["total_test"],
    "H1 2026 tests": channel_2026["total_test"],
    "H1 2025 patients": channel_2025["total_patients"],
    "H1 2026 patients": channel_2026["total_patients"],
    "H1 2025 V/P": channel_2025["visit_per_patient"],
    "H1 2026 V/P": channel_2026["visit_per_patient"],
    "H1 2025 T/V": channel_2025["test_per_visit"],
    "H1 2026 T/V": channel_2026["test_per_visit"]
})

channel_comparison

In [ ]:
result_channel = pd.DataFrame()
result_channel['H1 2025 tests'] = channel_comparison['H1 2025 tests']
result_channel['H1 2026 tests'] = channel_comparison['H1 2026 tests']

# Calculate percentage growth in tests
result_channel["Growth"] = ((channel_comparison["H1 2026 tests"] - channel_comparison["H1 2025 tests"])/ channel_comparison["H1 2025 tests"] * 100).round(1)

# Calculate percentage change in patients
result_channel["Patients"] = ((channel_comparison["H1 2026 patients"] - channel_comparison["H1 2025 patients"])/ channel_comparison["H1 2025 patients"] * 100).round(1)

# Calculate percentage change in V/P
result_channel["V/P"] = ((channel_comparison["H1 2026 V/P"] - channel_comparison["H1 2025 V/P"])/ channel_comparison["H1 2025 V/P"] * 100).round(1)

# Calculate percentage change in T/V
result_channel["T/V"] = ((channel_comparison["H1 2026 T/V"] - channel_comparison["H1 2025 T/V"]) / channel_comparison["H1 2025 T/V"] * 100).round(1)

result_channel

# H4 — Which city is short, and where most

For plan.csv

In [ ]:
# 1. Prepare the plan table
plan = df_plan.copy()

# Standardize column names
plan = plan.rename(columns={"City": "city"})

# 2. Check the plan columns
print("Plan columns:", plan.columns.tolist())
display(plan.head())


In [ ]:
# Define the H1 period (January–June 2026)
months = ["2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]

# Calculate the total H1 planned value for each city and metric
plan["plan_h1"] = df_plan[months].sum(axis=1)

# Validate the overall H1 plan by summing all city and metric totals
print("Total H1 plan:", plan["plan_h1"].sum())

# Check which metrics are included in the plan
print("Metrics:", plan["metric"].unique())

In [ ]:
# Make city a regular column if it is currently the index
actual = city_comparison.copy().reset_index()

# Standardize the city column
actual = actual.rename(columns={"City": "city"})

# Merge actual data with the plan
h4 = actual.merge(plan[["city", "plan_h1"]],on="city",how="left",validate="one_to_one",indicator=True)

In [ ]:
# Validate the join
print("Rows before join:", len(actual))
print("Rows after join:", len(h4))
print("Unmatched cities:")
display(h4.loc[h4["_merge"] != "both", ["city", "_merge"]])

# Remove merge indicator after validation
h4 = h4.drop(columns="_merge")

h4

In [ ]:
# Gap between plan and actual 2026 tests
h4["gap"] = h4["plan_h1"] - h4["H1 2026 tests"]

# Sort cities by gap, largest first
h4 = h4.sort_values("gap", ascending=False)

# Calculate the total row
total_plan = h4["plan_h1"].sum()
total_tests_2025 = h4["H1 2025 tests"].sum()
total_gap = h4["gap"].sum()

total_row = pd.DataFrame([{"city": "Total","plan_uplift": total_plan / total_tests_2025 - 1, "gap": total_gap,"share": 1.0}])

In [ ]:
# Calculate plan uplift for each city
h4["plan_uplift"] = (h4["plan_h1"] / h4["H1 2025 tests"] - 1)

# Calculate share of total gap
h4["share"] = h4["gap"] / total_gap

# Select and rename columns
city_result = h4[["city", "plan_uplift", "gap", "share"]].copy()

# Append Total row
city_result = pd.concat([city_result, total_row],ignore_index=True)

In [ ]:
# Format percentages and numbers
city_result["Plan uplift"] = city_result["plan_uplift"].map(lambda x: f"{x:+.1%}")
city_result["Gap (tests)"] = city_result["gap"].map(lambda x: f"{x:,.0f}")
city_result["Share of gap"] = city_result["share"].map(lambda x: f"{x:.1%}")

# Final presentation table
city_result = city_result[["city", "Plan uplift", "Gap (tests)", "Share of gap"]].rename(columns={"city": "City"})

city_result


In [ ]:
h1_data.head()

Step 1: Join the h1_data table with patients.csv

In [ ]:
tests = h1_data.merge(df_patients[["patient_id", "signup_date"]],on="patient_id",how="left",validate="many_to_one")

In [ ]:
# Validate the join
print("Total test records:", len(tests))
print("Missing signup dates:", tests["signup_date"].isna().sum())

In [ ]:
tests.head()

Step 2: Keep actual tests and classify patients

In [ ]:
tests = tests[tests["is_test"].astype(str).str.strip().str.lower().isin(["true", "1", "yes"])]

In [ ]:
# Keep the four growing cities
growing_cities = ["Ahmedabad","Bengaluru","Hyderabad","Pune"]

In [ ]:
tests = tests[tests["city"].isin(growing_cities)]

In [ ]:
# Convert year to numeric
tests["year"] = pd.to_numeric(tests["year"], errors="coerce")

In [ ]:
# Define new vs returning for each half-year
tests["patient_type"] = np.where(tests["signup_date"] >= pd.to_datetime(tests["year"].astype("Int64").astype(str) + "-01-01"),"New","Returning")

In [ ]:
tests[["city", "year", "patient_id", "signup_date", "patient_type"]].head()

Step 3: Calculate test counts by city, year and patient type

In [ ]:
# Count actual tests by city, year and patient type
test_counts = (tests.groupby(["city", "year", "patient_type"]).size().unstack(["year", "patient_type"], fill_value=0))

test_counts

Step 4: Calculate new-patient and returning-patient test growth

In [ ]:
# Calculate percentage growth from H1 2025 to H1 2026
growth_rows = []

for city in growing_cities:
    row = {"City": city}

    for patient_type in ["New", "Returning"]:
        tests_2025 = len(tests[(tests["city"] == city) & (tests["year"] == 2025) & (tests["patient_type"] == patient_type)])

        tests_2026 = len(tests[(tests["city"] == city) & (tests["year"] == 2026) & (tests["patient_type"] == patient_type)])

        growth = ((tests_2026 / tests_2025 - 1) * 100 if tests_2025 > 0 else np.nan)

        row[f"{patient_type}-patient tests"] = growth

    growth_rows.append(row)

growth_table = pd.DataFrame(growth_rows)

growth_table

Step 5: Add the plan uplift and format the final table

In [ ]:
# Get plan uplift for the four growing cities
plan_table = h4[["city","plan_h1","H1 2025 tests"]]

plan_table["Plan"] = (plan_table["plan_h1"] / plan_table["H1 2025 tests"] - 1) * 100

In [ ]:
print("Plan table:", plan_table.columns.tolist())
print("Growth table:", growth_table.columns.tolist())

In [ ]:
growth_table = growth_table.rename(columns={"City": "city"})

In [ ]:
plan_table = plan_table.rename(columns={"City": "city"})

In [ ]:
# Merge with new vs returning patient growth
final_h4 = growth_table.merge(plan_table[["city", "Plan"]],on="city",how="left",validate="one_to_one")

In [ ]:
# Format percentages
for col in ["New-patient tests","Returning-patient tests","Plan"]:
    final_h4[col] = final_h4[col].map(lambda x: f"{x:+.1f}%" if pd.notna(x) else "N/A")

final_h4

#H5 — Bookings per patient across months

In [ ]:
h1_data.head()

In [ ]:
h1_data['month'] = h1_data['slot_at'].dt.strftime('%m')

In [ ]:
h1_data.head()

In [ ]:
#get metric from the data
test_volume_month=h1_data.groupby(['year','month']).agg(total_patients = ('patient_id','nunique'),
                                        total_test = ('is_test','sum'),
                                        total_revenue = ('test_price','sum'),
                                        total_visits = ('booking_id','nunique')).reset_index()
test_volume_month

In [ ]:
test_volume_month['test_per_visit'] = (
    test_volume_month['total_test'] / test_volume_month['total_visits']
)

test_volume_month['visit_per_patient'] = (
    test_volume_month['total_visits'] / test_volume_month['total_patients']
)

test_volume_month

In [ ]:
# Create month-wise comparison for H1 2025 vs H1 2026
month_comparison = test_volume_month.pivot(index='month',columns='year',values=['total_test','total_patients','visit_per_patient','test_per_visit'])

In [ ]:
# Rename columns
month_comparison.columns = [f"H1 {year} {metric}"
    for metric, year in month_comparison.columns]

month_comparison = month_comparison.reset_index()

month_comparison

In [ ]:
#Half-year visits per patient
vp_2025 = test_volume.loc[test_volume['year'] == '2025', 'visit_per_patient'].iloc[0]
vp_2026 = test_volume.loc[test_volume['year'] == '2026', 'visit_per_patient'].iloc[0]

vp_change = ((vp_2026 - vp_2025) / vp_2025) * 100

In [ ]:
# Monthly year-on-year change in visits per patient
month_comparison['V/P change %'] = ((month_comparison['H1 2026 visit_per_patient'] -month_comparison['H1 2025 visit_per_patient'])/ month_comparison['H1 2025 visit_per_patient'] * 100)

monthly_min = month_comparison['V/P change %'].min()
monthly_max = month_comparison['V/P change %'].max()

In [ ]:
# City-level change in visits per patient
city_comparison['V/P change %'] = ((city_comparison['H1 2026 V/P'] -city_comparison['H1 2025 V/P'])/ city_comparison['H1 2025 V/P'] * 100)

In [ ]:
# Calculate H1 2025 patients returning in H1 2026
# Count patients present in both periods, divided by H1 2025 patients
patient_period = h1_data[h1_data['year'].isin([2025, 2026])].groupby(['year', 'city'])['patient_id'].unique()

In [ ]:
# Build the summary table
results = pd.DataFrame({'Check': ['Half-year V/P','Monthly year-on-year moves','Chennai','Jaipur','H1 2025 patients returning in H1 2026'],
                        'Finding': [f'{vp_2025:.4f} → {vp_2026:.4f} ({vp_change:+.1f}%)',f'Between {monthly_min:+.1f}% and {monthly_max:+.1f}%',
                                    f'{city_comparison.loc["Chennai", "V/P change %"]:+.1f}%',f'{city_comparison.loc["Jaipur", "V/P change %"]:+.1f}%',
                                    'Calculate using patient-level matching']})
results